# Динамическое программирование


**Динамическое программирование (DP)** применимо, когда задача удовлетворяет двум свойствам:

1. **Оптимальная подструктура** — решение задачи можно получить из решений её подзадач.
2. **Перекрывающиеся подзадачи** — одни и те же подзадачи возникают много раз.

Идея: посчитать каждую подзадачу **один раз** и сохранить результат.


## Два подхода

### Top-down (memoization)
Пишем рекурсию «сверху», но кэшируем ответы. Хорошо подходит, когда не все подзадачи нужны.

### Bottom-up (tabulation)
Заполняем таблицу `dp[]` от меньших подзадач к большим. Экономит память на стеке рекурсии.


## Классика: Фибоначчи


In [ ]:
# Top-down с functools.lru_cache
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_topdown(n):
    if n < 2: return n
    return fib_topdown(n - 1) + fib_topdown(n - 2)

fib_topdown(50)

In [ ]:
# Bottom-up с массивом
def fib_bottomup(n):
    if n < 2: return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

fib_bottomup(50)

In [ ]:
# Bottom-up O(1) по памяти
def fib_optimized(n):
    if n < 2: return n
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

fib_optimized(50)

## Классика: лестница (Climbing stairs)
За шаг можно сделать 1 или 2 ступеньки. Сколько способов подняться на n ступенек?

По сути — Фибоначчи: `dp[n] = dp[n-1] + dp[n-2]`.


In [ ]:
def climb_stairs(n):
    if n <= 2: return n
    a, b = 1, 2
    for _ in range(n - 2):
        a, b = b, a + b
    return b

climb_stairs(5)   # 8

## Классика: монеты (Coin Change)
Дан набор номиналов монет и сумма `amount`. Найти минимальное число монет, чтобы её набрать.

`dp[i]` — минимальное число монет для суммы `i`.


In [ ]:
def coin_change(coins, amount):
    INF = amount + 1
    dp = [INF] * (amount + 1)
    dp[0] = 0
    for i in range(1, amount + 1):
        for c in coins:
            if c <= i:
                dp[i] = min(dp[i], dp[i - c] + 1)
    return dp[amount] if dp[amount] != INF else -1

coin_change([1, 2, 5], 11)   # 3 (5+5+1)

## Классика: 0/1 Knapsack (Рюкзак)
Дано: `n` предметов с весами `w[i]` и стоимостями `v[i]`. Рюкзак вмещает `W`.
Каждый предмет можно взять **0 или 1 раз**. Максимизировать суммарную стоимость.

`dp[i][w]` — макс. стоимость, используя первые `i` предметов в рюкзаке весом `w`.


In [ ]:
def knapsack(weights, values, W):
    n = len(weights)
    dp = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(W + 1):
            dp[i][w] = dp[i - 1][w]                    # не берём i-й предмет
            if weights[i - 1] <= w:
                dp[i][w] = max(
                    dp[i][w],
                    dp[i - 1][w - weights[i - 1]] + values[i - 1]
                )
    return dp[n][W]

weights = [1, 3, 4, 5]
values  = [1, 4, 5, 7]
knapsack(weights, values, 7)   # 9 (предметы 2 + 3: вес 3+4=7, стоимость 4+5=9)

## Классика: LCS (Longest Common Subsequence)
Найти длину наибольшей общей подпоследовательности двух строк.

`dp[i][j]` — длина LCS для первых `i` символов `s1` и `j` символов `s2`.


In [ ]:
def lcs(s1, s2):
    n, m = len(s1), len(s2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

lcs('ABCBDAB', 'BDCAB')   # 4 (например, 'BCAB')

## Классика: LIS (Longest Increasing Subsequence)
Длиннейшая строго возрастающая подпоследовательность.

Простая версия — `O(n²)`:


In [ ]:
def lis(arr):
    n = len(arr)
    if n == 0: return 0
    dp = [1] * n
    for i in range(1, n):
        for j in range(i):
            if arr[j] < arr[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

lis([10, 9, 2, 5, 3, 7, 101, 18])   # 4 ([2, 3, 7, 101])

## Как распознать DP-задачу на собесе

Триггеры в формулировке:
- «найти **минимальное / максимальное** ...»
- «сколько **способов** ...»
- «есть ли способ ...» (часто можно через DP/BFS)
- последовательность шагов с выбором

Шаги решения:
1. Определить **состояние** (`dp[i]`, `dp[i][j]`).
2. Найти **переход** (рекуррентное соотношение).
3. Базовые случаи.
4. Порядок заполнения.
5. Подумать, можно ли уменьшить память (часто хватает 1-2 предыдущих строк).
